# Recipe 01 — Multimodal Converter
> Convert between MultiModalItem and ContextItem using pluggable encoders.

| | |
|---|---|
| **Module** | `anchor.multimodal` |
| **Key classes** | `MultiModalConverter`, `MultiModalItem`, `MultiModalContent`, `ModalityType` |
| **Difficulty** | Beginner |

In [ ]:
# Setup
from anchor.multimodal import (
    MultiModalConverter,
    MultiModalItem,
    MultiModalContent,
    ModalityType,
)
from anchor.multimodal.encoders import TextEncoder, CompositeEncoder
from anchor.models import ContextItem, SourceType

## Walkthrough

In [ ]:
# 1 — Create a multimodal item
# MultiModalContent wraps raw content with its modality type
content = MultiModalContent(
    modality=ModalityType.TEXT,
    content="A document about AI safety and alignment research",
)
print(f"Modality : {content.modality}")
print(f"Content  : {content.content}")

In [ ]:
# MultiModalItem bundles one or more contents with metadata
item = MultiModalItem(
    id="mm-1",
    contents=[content],
    source="wiki",
    score=0.9,
    priority=5,
    metadata={"author": "research-team"},
)

print(f"Item ID   : {item.id}")
print(f"Source    : {item.source}")
print(f"Score     : {item.score}")
print(f"Priority  : {item.priority}")
print(f"Contents  : {len(item.contents)}")
print(f"Metadata  : {item.metadata}")

In [ ]:
# 2 — Set up encoders and convert to ContextItem
# TextEncoder handles ModalityType.TEXT
text_encoder = TextEncoder()
print(f"Encoder: {type(text_encoder).__name__}")

# CompositeEncoder routes each modality to the right encoder
composite = CompositeEncoder(
    encoders={ModalityType.TEXT: text_encoder},
)
print(f"Composite handles: {list(composite.encoders.keys())}")

In [ ]:
# Convert a single MultiModalItem -> ContextItem
context_item = MultiModalConverter.to_context_item(item, encoder=composite)

print(f"Type       : {type(context_item).__name__}")
print(f"Content    : {context_item.content[:60]}...")
print(f"Source type: {context_item.source_type}")

In [ ]:
# 3 — Batch conversion
# Create several multimodal items
items = [
    MultiModalItem(
        id=f"mm-{i}",
        contents=[MultiModalContent(modality=ModalityType.TEXT, content=f"Document {i}")],
        source="wiki",
        score=0.8 + i * 0.05,
        priority=i,
        metadata={},
    )
    for i in range(1, 4)
]

# Batch convert
context_items = MultiModalConverter.to_context_items(items, encoder=composite)

print(f"Converted {len(context_items)} items:")
for ci in context_items:
    print(f"  {ci.content}")

In [ ]:
# Inspect ContextItem attributes
for ci in context_items:
    print(f"  source_type={ci.source_type}  content_len={len(ci.content)}")

In [ ]:
# 4 — Round-trip: convert back to MultiModalItem
# Convert a ContextItem back to a MultiModalItem
mm_item = MultiModalConverter.from_context_item(
    context_item,
    modality=ModalityType.TEXT,
)

print(f"Recovered ID      : {mm_item.id}")
print(f"Recovered modality: {mm_item.contents[0].modality}")
print(f"Recovered content : {mm_item.contents[0].content[:60]}...")

In [ ]:
# Verify round-trip fidelity
original_text = item.contents[0].content
recovered_text = mm_item.contents[0].content
print(f"Original  : {original_text}")
print(f"Recovered : {recovered_text}")
print(f"Match     : {original_text == recovered_text}")

In [ ]:
# Enumerate all supported modality types
for modality in ModalityType:
    print(f"  {modality.name}: {modality.value}")

## Key Takeaways
- `MultiModalContent` pairs raw data with a `ModalityType` tag.
- `MultiModalItem` groups contents with scoring and metadata.
- `CompositeEncoder` dispatches encoding to modality-specific encoders.
- `MultiModalConverter.to_context_item()` and `from_context_item()` enable round-trip conversion.
- Batch conversion via `to_context_items()` handles multiple items efficiently.

**Next:** [Image Encoding](02_image_encoding.ipynb)